In [1]:
!pip install --upgrade git+https://github.com/yhilpisch/tpqoa.git


  Cloning https://github.com/yhilpisch/tpqoa.git to c:\users\bedla\appdata\local\temp\pip-req-build-bm2ay4ch


  ERROR: Error [WinError 2] Das System kann die angegebene Datei nicht finden while executing command git version
ERROR: Cannot find command 'git' - do you have 'git' installed and in your PATH?


In [4]:
!pip install mplfinance

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 954.4 kB/s eta 0:00:003 MB/s eta 0:00:01


In [2]:
import tpqoa
import pandas as pd
import mplfinance as mpf
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
import datetime
import time
from collections import deque

# Konfiguration
api = tpqoa.tpqoa(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Accounts\oanda.cfg')
instrument = "EUR_USD"
candle_seconds = 5  # Kerzendauer in Sekunden
lookback_candles = 40  # Anzahl anzuzeigender Kerzen
chart_update_every_n_ticks = 10  # Chart nach X Ticks aktualisieren

class CandleBuilder:
    """Baut Kerzen aus Streaming-Daten"""
    
    def __init__(self, candle_seconds=5, max_candles=100):
        self.candle_seconds = candle_seconds
        self.max_candles = max_candles
        self.candles = deque(maxlen=max_candles)
        self.current_candle = None
        self.current_candle_start = None
        self.last_price = None
        self.last_bid = None
        self.last_ask = None
        self.tick_count = 0
        self.error_count = 0
        self.successful_ticks = 0
        
    def add_tick(self, timestamp_str, bid, ask):
        """Fügt einen neuen Tick hinzu"""
        try:
            self.tick_count += 1
            
            # Debug für erste Ticks
            if self.tick_count <= 3:
                print(f"\n✓ Tick #{self.tick_count} wird verarbeitet:")
                print(f"  Timestamp: {timestamp_str}")
                print(f"  Bid: {bid}, Ask: {ask}")
            
            # Preis berechnen
            self.last_bid = bid
            self.last_ask = ask
            mid_price = (bid + ask) / 2
            self.last_price = mid_price
            
            if self.tick_count <= 3:
                print(f"  Mid-Preis: {mid_price:.5f}")
            
            # Timestamp parsen
            dt = pd.to_datetime(timestamp_str)
            
            if self.tick_count <= 3:
                print(f"  Parsed Datetime: {dt}")
            
            # Kerzenstart berechnen
            candle_start = dt.floor(f'{self.candle_seconds}s')
            
            if self.tick_count <= 3:
                print(f"  Candle Start: {candle_start}")
            
            new_candle_started = False
            
            # Neue Kerze beginnen?
            if self.current_candle_start is None or candle_start > self.current_candle_start:
                # Alte Kerze speichern
                if self.current_candle is not None:
                    self.candles.append({
                        'timestamp': self.current_candle_start,
                        'open': self.current_candle['open'],
                        'high': self.current_candle['high'],
                        'low': self.current_candle['low'],
                        'close': self.current_candle['close'],
                        'volume': self.current_candle['volume']
                    })
                    new_candle_started = True
                    print(f"\n{'='*60}")
                    print(f"✓ KERZE #{len(self.candles)} ABGESCHLOSSEN!")
                    print(f"  Zeit: {self.current_candle_start}")
                    print(f"  Open:   {self.current_candle['open']:.5f}")
                    print(f"  High:   {self.current_candle['high']:.5f}")
                    print(f"  Low:    {self.current_candle['low']:.5f}")
                    print(f"  Close:  {self.current_candle['close']:.5f}")
                    print(f"  Volume: {self.current_candle['volume']} Ticks")
                    print(f"{'='*60}\n")
                
                # Neue Kerze starten
                self.current_candle_start = candle_start
                self.current_candle = {
                    'open': mid_price,
                    'high': mid_price,
                    'low': mid_price,
                    'close': mid_price,
                    'volume': 1
                }
                
                if self.tick_count <= 3:
                    print(f"→ Neue Kerze gestartet um {candle_start}\n")
            else:
                # Aktuelle Kerze updaten
                if self.current_candle is not None:
                    self.current_candle['high'] = max(self.current_candle['high'], mid_price)
                    self.current_candle['low'] = min(self.current_candle['low'], mid_price)
                    self.current_candle['close'] = mid_price
                    self.current_candle['volume'] += 1
            
            self.successful_ticks += 1
            return new_candle_started
            
        except Exception as e:
            self.error_count += 1
            if self.error_count <= 5:
                print(f"\n⚠ FEHLER bei Tick #{self.tick_count}: {e}")
            return False
    
    def get_dataframe(self):
        """Gibt DataFrame mit allen Kerzen zurück"""
        data = list(self.candles)
        
        # Aktuelle Kerze hinzufügen
        if self.current_candle is not None and self.current_candle_start is not None:
            data.append({
                'timestamp': self.current_candle_start,
                'open': self.current_candle['open'],
                'high': self.current_candle['high'],
                'low': self.current_candle['low'],
                'close': self.current_candle['close'],
                'volume': self.current_candle['volume']
            })
        
        if not data:
            return None
        
        df = pd.DataFrame(data)
        df.set_index('timestamp', inplace=True)
        df.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
        return df

def draw_chart(df, iteration, candle_builder):
    """Zeichnet den Chart"""
    clear_output(wait=True)
    
    print(f"{'='*70}")
    print(f"OANDA Live Chart - {instrument}")
    print(f"Kerzendauer: {candle_seconds}s | Anzeige: {lookback_candles} Kerzen")
    print(f"{'='*70}")
    print(f"Update #{iteration} | {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Kerzen: {len(df)} (vollständig: {len(candle_builder.candles)})")
    print(f"Ticks: {candle_builder.successful_ticks} erfolgreich | {candle_builder.error_count} Fehler")
    if candle_builder.last_price:
        print(f"Preis: {candle_builder.last_price:.5f} (Bid/Ask: {candle_builder.last_bid:.5f}/{candle_builder.last_ask:.5f})")
    if candle_builder.current_candle:
        print(f"Aktuelle Kerze: {candle_builder.current_candle['volume']} Ticks")
    print(f"{'='*70}\n")
    
    try:
        if len(df) >= 2:
            # Y-Achsen-Grenzen berechnen für bessere Darstellung
            price_min = df[['Low']].min().min()
            price_max = df[['High']].max().max()
            price_range = price_max - price_min
            margin = price_range * 0.1  # 10% Margin oben und unten
            
            fig, axes = mpf.plot(
                df.tail(lookback_candles),
                type='candle',
                style='charles',
                volume=False,
                returnfig=True,
                ylabel='Preis (USD)',
                figsize=(14, 7),
                ylim=(price_min - margin, price_max + margin)
            )
            
            # Y-Achsen-Format korrigieren
            ax = axes[0]
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:.5f}'))
            
            plt.suptitle(f'{instrument} - {candle_seconds}s Kerzen (Live)', fontsize=16, y=0.995)
            display(fig)
            plt.close(fig)
        else:
            print("Warte auf mindestens 2 abgeschlossene Kerzen...")
    except Exception as e:
        print(f"Chart-Fehler: {e}")

def on_success(instrument_name, time, bid, ask):
    """Callback-Funktion für jeden Tick"""
    global candle_builder, chart_iteration, ticks_since_last_chart
    
    # Tick verarbeiten
    new_candle = candle_builder.add_tick(time, bid, ask)
    ticks_since_last_chart += 1
    
    # Chart aktualisieren?
    should_update = (
        new_candle or
        ticks_since_last_chart >= chart_update_every_n_ticks or
        candle_builder.successful_ticks <= 5
    )
    
    if should_update:
        df = candle_builder.get_dataframe()
        
        if df is not None and len(df) > 0:
            chart_iteration += 1
            draw_chart(df, chart_iteration, candle_builder)
            ticks_since_last_chart = 0

def start_live_chart():
    """Hauptfunktion"""
    global candle_builder, chart_iteration, ticks_since_last_chart
    
    print("="*70)
    print("OANDA Live Chart mit Streaming-Daten")
    print("="*70)
    print(f"Instrument: {instrument}")
    print(f"Kerzendauer: {candle_seconds} Sekunden")
    print(f"Anzeige: {lookback_candles} Kerzen")
    print(f"Chart-Update: alle {chart_update_every_n_ticks} Ticks oder bei neuer Kerze")
    print("="*70)
    print("\nStarte Stream... (erste 2-3 Kerzen brauchen 10-15 Sekunden)")
    print("Drücke Strg+C zum Beenden\n")
    
    # Globals initialisieren
    candle_builder = CandleBuilder(candle_seconds=candle_seconds, max_candles=lookback_candles * 2)
    chart_iteration = 0
    ticks_since_last_chart = 0
    
    try:
        # Stream mit Callback starten
        api.stream_data(instrument, stop=None, callback=on_success)
        
    except KeyboardInterrupt:
        print("\n\nBeendet durch Benutzer")
        print(f"\n{'='*70}")
        print("STATISTIK:")
        print(f"{'='*70}")
        print(f"  Erfolgreiche Ticks: {candle_builder.successful_ticks}")
        print(f"  Fehlerhafte Ticks:  {candle_builder.error_count}")
        # print(f"  Kerzen erstellt:    {len(candle_builder.candles)}")
        print(f"  Chart-Updates:      {chart_iteration}")
        print(f"{'='*70}")
    except Exception as e:
        print(f"\n\nFEHLER: {e}")
        import traceback
        traceback.print_exc()

# Globals für Callback
candle_builder = None
chart_iteration = 0
ticks_since_last_chart = 0

# Programm starten
if __name__ == "__main__":
    start_live_chart()



Beendet durch Benutzer

STATISTIK:
  Erfolgreiche Ticks: 1
  Fehlerhafte Ticks:  0
  Chart-Updates:      1
